# Predição de Churn em Telecomunicações — Entrega Acadêmica

**Aprendizado supervisionado · classificação binária · pipeline de ML Engineering**

Autor: **Vinicius Gomes**

---

## Objetivo deste notebook

Este notebook é a **narrativa acadêmica completa e auto-contida** de um sistema de
predição de churn: da aquisição dos dados até o desenho de monitoramento.

Ele responde, na ordem, às perguntas que definem o trabalho:

1. Que problema estamos resolvendo, e por que ele não é "maximizar acurácia"?
2. De onde vêm os dados, e como sabemos que são exatamente os dados certos?
3. Como o conjunto de teste foi protegido de vazamento?
4. Quais modelos foram considerados, e como foram comparados?
5. Por que o modelo final é uma regressão logística?
6. Qual limiar de decisão foi adotado, e o que ele custa?
7. Quão bom o modelo é de fato, no conjunto que nunca participou de decisão alguma?
8. Como cada previsão é explicada?
9. Como o sistema seria monitorado em produção?

### Duas propriedades que este notebook garante

**Não decide nada.** Toda decisão de modelagem — modelo, features, hiperparâmetros,
política de calibração, limiar — já foi tomada e congelada no repositório de
engenharia. Aqui elas são **reproduzidas e verificadas**, nunca revisitadas. Nenhuma
configuração é escolhida depois de olhar um resultado.

```
academic_reproduction_only     = True
evaluated_configurations       = 1
selection_after_reproduction   = False
model_changed                  = False
```

**Não depende de nada local.** Roda em um runtime novo do Google Colab, sem upload
manual, sem credencial, sem caminho absoluto e sem o repositório no disco. Os dados
são baixados de uma fonte pública e a integridade é conferida por SHA-256 antes de
qualquer análise.

### Relação com o repositório de engenharia

| | |
|---|---|
| **Repositório** | fonte da verdade de produção/engenharia: pipeline persistido, API, testes, monitoramento |
| **Este notebook** | reprodução acadêmica auto-contida do protocolo congelado |

Os dois devem produzir os **mesmos números**. Cada valor reproduzido aqui é comparado
com o valor registrado no artefato versionado correspondente, e a diferença é
mostrada explicitamente. Nada é afirmado sem ser conferido na tela.

## 1. Contexto e descrição do problema

### O problema de negócio

Churn é a saída de um cliente para um concorrente ou o cancelamento do serviço. Em
telecomunicações o produto é uma assinatura recorrente: a receita não vem de uma
venda, vem da **permanência**. Adquirir um cliente novo custa consistentemente mais
do que reter um existente, o que torna a pergunta operacional bastante concreta:

> Quais clientes estão em risco de sair **enquanto ainda é possível agir**?

### O problema estatístico

Tarefa de **classificação binária supervisionada**:

| | |
|---|---|
| Classe positiva | `Churn = Yes` — o cliente saiu |
| Classe negativa | `Churn = No` — o cliente permaneceu |
| Saída primária | probabilidade de churn (score contínuo em `[0, 1]`) |
| Saída secundária | decisão binária, obtida aplicando um limiar justificado ao score |

A saída primária ser a probabilidade, e não a classe, é uma decisão de projeto: uma
operação de retenção precisa **priorizar** uma carteira, e priorizar exige ordenar.
O limiar é uma segunda decisão, separada, tomada depois — e discutida na seção 9.

### Por que acurácia não pode ser o critério

A base é desbalanceada: cerca de **26,5 %** dos clientes saíram. Um classificador
trivial que responde "ninguém sai" para todo mundo acerta **73,5 %** dos casos e é
**completamente inútil** — não encontra um único cliente em risco.

Acurácia, sozinha, não distingue esse modelo de um modelo útil. Por isso o critério
primário deste trabalho é **Average Precision** (área sob a curva Precision-Recall),
que mede qualidade de ordenação na classe minoritária, com **ROC-AUC** como métrica
secundária. Acurácia aparece, mas como métrica **auxiliar** — nunca como manchete.

## 2. Ambiente e dependências

O notebook usa apenas a base científica do Python. Nada de FastAPI, servidor web ou
framework de testes: essas peças pertencem ao produto de engenharia, não à narrativa
acadêmica.

A célula abaixo instala as dependências **somente se estiverem ausentes**, o que
torna a execução no Colab rápida (o Colab já traz todas) sem deixar de funcionar em
um ambiente vazio.

In [ ]:
# Instala apenas o que faltar. No Colab, normalmente nada é instalado.
import importlib
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.2",
    "sklearn": "scikit-learn>=1.5",
    "matplotlib": "matplotlib>=3.8",
}

missing = [spec for mod, spec in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Instalando:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Todas as dependências já estão presentes.")

In [ ]:
import hashlib
import io
import platform
import urllib.request

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["figure.figsize"] = (7.2, 4.2)
matplotlib.rcParams["axes.grid"] = True
matplotlib.rcParams["grid.alpha"] = 0.25
pd.set_option("display.width", 110)

ENVIRONMENT = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
}
print("Ambiente de execução")
for name, version in ENVIRONMENT.items():
    print(f"  {name:>14s} : {version}")

### Uma observação sobre versões

As bibliotecas do Colab evoluem. Este notebook **não assume** que a versão do
scikit-learn é idêntica à que gerou os artefatos do repositório — ele **compara os
resultados** e mostra a diferença. Se um número divergir, a divergência aparece na
tela em vez de passar despercebida.

O protocolo é o que está congelado: a partição, a semente, o pré-processamento, o
estimador e o limiar. Esses são reproduzidos exatamente.

## 3. Aquisição dos dados

### Origem

O conjunto é o **Telco Customer Churn**, dado de exemplo publicado pela IBM e
amplamente usado como benchmark de churn. O repositório de engenharia registra a
página do Kaggle (`blastchar/telco-customer-churn`) como origem canônica — mas o
Kaggle exige autenticação, o que quebraria a exigência de reprodutibilidade sem
credenciais.

Este notebook usa, por isso, o **repositório público da própria IBM** no GitHub, que
serve o mesmo arquivo por HTTPS, sem conta, sem token e sem cabeçalho de autorização.

### Duas identidades diferentes, e por que a distinção importa

O mirror público **não** entrega os mesmos bytes que o projeto congelou, e seria
incorreto afirmar que entrega. A diferença é de **serialização**, não de conteúdo:

| | Representação | Terminador de linha |
|---|---|---|
| Bytes servidos pelo mirror | `LF` | `\n` |
| Arquivo congelado no repositório | `CRLF` | `\r\n` |

São a mesma tabela, gravada de duas maneiras. Por isso o notebook calcula e registra
**dois digests distintos**, com nomes distintos:

* `downloaded_bytes_sha256` — SHA-256 dos bytes **exatamente como recebidos** da URL.
  É uma propriedade do mirror, não do projeto, e **não** precisa coincidir com nada.
* `canonicalized_dataset_sha256` — SHA-256 **depois** de reserializar as quebras de
  linha na forma canônica do projeto. É este, e somente este, que tem de bater com a
  constante congelada.

### A regra de canonicalização, por extenso

A transformação toca **exclusivamente a representação de quebra de linha**:

1. o fluxo de bytes é dividido em linhas lógicas nos terminadores presentes
   (`CRLF`, `CR` isolado ou `LF` isolado);
2. as linhas são reserializadas com `CRLF`.

Nada mais é tocado. Não há remoção de espaços, reordenação de linhas ou colunas,
conversão de valores, normalização de números, alteração de aspas ou mudança
semântica de codificação. A célula abaixo **prova** isso: removidos todos os
terminadores de linha, os dois fluxos de bytes são idênticos.

Essa prova é o ponto. Sem ela, "normalizar até o hash bater" seria indistinguível de
massagear um dataset diferente até que ele passasse na verificação.

### E se não bater

A célula **aborta**. Nenhuma análise roda sobre dados que não sejam exatamente os
dados aprovados — e a verificação de esquema logo abaixo confirma, de forma
independente do hash, que o que chegou é a tabela esperada.

In [ ]:
# --- Origem pública, sem autenticação -------------------------------------
DATASET_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

# Digest congelado da REPRESENTAÇÃO CANÔNICA do projeto (linhas terminadas em CRLF).
# Não é o digest dos bytes servidos pelo mirror, e não deveria ser.
FROZEN_CANONICAL_SHA256 = "88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a"

CANONICAL_LINE_TERMINATOR = b"\r\n"

# As 21 colunas esperadas, na ordem em que o arquivo aprovado as traz.
EXPECTED_COLUMNS = [
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
]
EXPECTED_N_ROWS = 7043


def canonicalise_newlines(payload: bytes) -> bytes:
    """Reserializa o conteúdo com o terminador de linha canônico do projeto.

    A regra, e apenas ela:

    1. divide o fluxo em linhas lógicas nos terminadores presentes (CRLF, CR ou LF);
    2. reserializa essas linhas com CRLF.

    Nada mais é tocado: nenhum espaço removido, nenhuma linha ou coluna reordenada,
    nenhum valor convertido, nenhum número normalizado, nenhuma aspa alterada, e os
    bytes dentro de cada linha preservados exatamente.
    """
    logical_lines = payload.replace(b"\r\n", b"\n").replace(b"\r", b"\n").split(b"\n")
    return CANONICAL_LINE_TERMINATOR.join(logical_lines)


def without_line_terminators(payload: bytes) -> bytes:
    """O conteúdo sem nenhum terminador de linha — a parte que não pode mudar."""
    return payload.replace(b"\r\n", b"").replace(b"\r", b"").replace(b"\n", b"")


def fetch_raw_dataset(url: str = DATASET_URL) -> bytes:
    """Baixa o CSV bruto. Sem credencial, sem cabeçalho de autorização."""
    with urllib.request.urlopen(url, timeout=120) as response:
        if response.status != 200:
            raise RuntimeError(f"A origem respondeu HTTP {response.status}.")
        return response.read()


# --- Identidade 1: os bytes como chegaram ---------------------------------
downloaded_bytes = fetch_raw_dataset()
downloaded_bytes_sha256 = hashlib.sha256(downloaded_bytes).hexdigest()

# --- Identidade 2: o conteúdo na representação canônica do projeto ---------
raw_bytes = canonicalise_newlines(downloaded_bytes)
canonicalized_dataset_sha256 = hashlib.sha256(raw_bytes).hexdigest()

print("Identidade 1 - bytes exatamente como servidos pelo mirror")
print(f"  tamanho                 : {len(downloaded_bytes):,} bytes")
print(f"  downloaded_bytes_sha256 : {downloaded_bytes_sha256}")
print("  (propriedade do mirror; nao precisa coincidir com nada do projeto)")

print("\nIdentidade 2 - mesmo conteudo, representacao canonica do projeto (CRLF)")
print(f"  tamanho                      : {len(raw_bytes):,} bytes")
print(f"  canonicalized_dataset_sha256 : {canonicalized_dataset_sha256}")
print(f"  frozen esperado              : {FROZEN_CANONICAL_SHA256}")

print(
    f"\nOs dois digests sao diferentes: {downloaded_bytes_sha256 != canonicalized_dataset_sha256}"
)
print(
    f"Diferenca de tamanho: {len(raw_bytes) - len(downloaded_bytes):,} bytes "
    f"= 1 byte por linha logica"
)

# --- A transformacao tocou apenas as quebras de linha ---------------------
newline_only = without_line_terminators(downloaded_bytes) == without_line_terminators(raw_bytes)
print(f"\nCanonicalizacao restrita a quebras de linha: {newline_only}")
if not newline_only:
    raise SystemExit(
        "ABORTADO: a canonicalizacao alterou algo alem das quebras de linha. "
        "Uma transformacao que muda conteudo nao esta compensando serializacao."
    )

# --- Somente agora o digest canonico e comparado com o congelado ----------
if canonicalized_dataset_sha256 != FROZEN_CANONICAL_SHA256:
    raise SystemExit(
        "ABORTADO: o conteudo canonicalizado nao e o dataset aprovado do projeto.\n"
        "Nenhuma analise pode ser executada sobre dados nao verificados."
    )

print("\nOK - integridade confirmada. Este e o dataset congelado do projeto.")

In [ ]:
# Verificacao independente do hash: o que chegou tem a forma esperada.
# Um digest prova identidade de bytes; isto prova que a tabela e a tabela certa,
# e as duas evidencias falham de maneiras diferentes.
_probe = pd.read_csv(io.BytesIO(raw_bytes))

print(f"linhas   : {_probe.shape[0]:,}   esperado: {EXPECTED_N_ROWS:,}")
print(f"colunas  : {_probe.shape[1]}   esperado: {len(EXPECTED_COLUMNS)}")
_schema_ok = list(_probe.columns) == EXPECTED_COLUMNS
print(f"esquema  : {'identico, na ordem esperada' if _schema_ok else 'DIVERGENTE'}")

assert _probe.shape[0] == EXPECTED_N_ROWS, "contagem de linhas inesperada"
assert _schema_ok, "esquema inesperado"
del _probe

> **Duas evidências independentes.** O digest responde "estes são os bytes
> aprovados?"; a verificação de esquema responde "esta é a tabela esperada?". Elas
> falham de maneiras diferentes — um arquivo truncado quebra a primeira, um arquivo de
> outra versão do dataset quebraria a segunda — e é por isso que ambas existem.

> **Por que abortar em vez de avisar.** Se o arquivo mudar na origem, todo número
> deste notebook passaria a descrever outra tabela, enquanto o texto continuaria
> afirmando os valores antigos. Um relatório que não pode ser rastreado até bytes
> específicos não é reprodutível — é apenas plausível. Abortar é a única resposta
> honesta.

## 4. Entendimento do dataset

Antes de qualquer transformação: o que existe na tabela, em que tipo, com que
domínio, e onde estão os problemas.

In [ ]:
ID_COLUMN = "customerID"
TARGET_COLUMN = "Churn"
POSITIVE_LABEL = "Yes"

raw = pd.read_csv(io.BytesIO(raw_bytes))

print(f"Dimensões: {raw.shape[0]} linhas × {raw.shape[1]} colunas")
print(
    f"Identificadores únicos: {raw[ID_COLUMN].nunique()} "
    f"({'sem duplicatas' if raw[ID_COLUMN].nunique() == len(raw) else 'HÁ DUPLICATAS'})"
)
print(f"Linhas duplicadas: {int(raw.duplicated().sum())}")
print(f"Valores nulos declarados: {int(raw.isna().sum().sum())}")

print("\nDistribuição da variável-alvo:")
counts = raw[TARGET_COLUMN].value_counts()
for label, n in counts.items():
    print(f"  {label:>4s} : {n:5d}  ({n / len(raw):6.2%})")

prevalence = float((raw[TARGET_COLUMN] == POSITIVE_LABEL).mean())
print(f"\nPrevalência da classe positiva : {prevalence:.4f}")
print(
    f"Acurácia da classe majoritária : {1 - prevalence:.4f}  <- a barra que acurácia precisa vencer"
)

In [ ]:
# O problema de qualidade que importa: TotalCharges chega como texto.
print(f"dtype de TotalCharges: {raw['TotalCharges'].dtype}  <- texto, não número")

blank_mask = raw["TotalCharges"].astype("string").str.strip().eq("")
print(f"Células em branco em TotalCharges: {int(blank_mask.sum())}")

print("\nEssas linhas, com tenure ao lado:")
display(raw.loc[blank_mask, [ID_COLUMN, "tenure", "MonthlyCharges", "TotalCharges", TARGET_COLUMN]])

print(f"\ntenure dessas linhas: {sorted(raw.loc[blank_mask, 'tenure'].unique())}")
print("Todas têm tenure == 0: são clientes que não fecharam o primeiro ciclo de faturamento.")

### O que as 11 células em branco significam

Elas **não** são dados faltantes no sentido usual. Todas as 11 linhas têm
`tenure == 0`: são clientes que ainda não completaram um ciclo de faturamento, então
o total acumulado ainda não existe. O valor correto é **zero**, e zero é um fato
sobre o cliente, não uma imputação.

Isso produz a regra que o projeto congelou:

| Situação | Tratamento | Justificativa |
|---|---|---|
| Branco **e** `tenure == 0` | **zero estrutural** | não houve faturamento ainda; zero é o valor verdadeiro |
| Branco **e** `tenure > 0` | **erro — rejeita** | um cliente com histórico tem de ter total; imputar aqui inventaria histórico |
| Não numérico | **erro — rejeita** | não é interpretável |

A segunda linha da tabela é a que importa metodologicamente: a regra **não** é
"preencher branco com zero". É "preencher com zero **onde zero é justificável**", e
falhar alto onde não é. Uma imputação silenciosa aqui esconderia corrupção de dados
em produção — e este mesmo transformador roda na API.

In [ ]:
# Um retrato compacto dos 19 preditores, para saber com o que estamos lidando.
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    c for c in raw.columns if c not in NUMERIC_FEATURES + [ID_COLUMN, TARGET_COLUMN]
]

print(
    f"Preditores: {len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)}"
    f"  ({len(NUMERIC_FEATURES)} numéricos, {len(CATEGORICAL_FEATURES)} categóricos)"
)
print("\ncustomerID é excluído: é identificador, não preditor.")

print("\nCardinalidade das categóricas:")
for column in CATEGORICAL_FEATURES:
    levels = raw[column].unique()
    shown = ", ".join(map(str, sorted(map(str, levels))[:4]))
    more = "" if len(levels) <= 4 else f", … (+{len(levels) - 4})"
    print(f"  {column:20s} {len(levels):2d}  [{shown}{more}]")

### O que chama atenção nesta tabela

**`SeniorCitizen` é numérica na origem mas categórica no significado**: é um
indicador 0/1, não uma quantidade. O projeto a trata como categórica.

**Vários "níveis" são na verdade ausência de serviço.** `OnlineSecurity` tem
`No internet service` como nível distinto de `No`. Isso não é ruído de codificação:
é uma dependência estrutural que reaparece — de forma importante — na explicabilidade
(seção 12) e no monitoramento (seção 14).

## 5. Análise exploratória — o suficiente para orientar decisões

A EDA completa está no repositório (`reports/eda_report.md`, 16 figuras). Aqui
reproduzimos apenas o que sustenta decisões posteriores.

> **Nota metodológica importante, e desconfortável.** A EDA original foi feita
> **antes** de o holdout ser formalmente protegido, sobre as 7043 linhas. Isso está
> registrado como uma limitação real do trabalho e é retomado na seção 11. Os
> gráficos abaixo servem para explicar o problema — não para escolher nada.

In [ ]:
churned = raw[TARGET_COLUMN] == POSITIVE_LABEL

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))

# (a) Alvo
counts = raw[TARGET_COLUMN].value_counts()
axes[0].bar(["No", "Yes"], [counts["No"], counts["Yes"]], color=["#4C78A8", "#E45756"])
axes[0].set_title("Distribuicao do alvo")
axes[0].set_ylabel("clientes")
for i, v in enumerate([counts["No"], counts["Yes"]]):
    axes[0].text(i, v, f"{v}\n{v / len(raw):.1%}", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(0, counts.max() * 1.20)

# (b) Churn por tipo de contrato
by_contract = (
    raw.groupby("Contract", observed=True)[TARGET_COLUMN]
    .apply(lambda s: (s == POSITIVE_LABEL).mean())
    .sort_values(ascending=False)
)
axes[1].barh(by_contract.index, by_contract.to_numpy(), color="#E45756")
axes[1].set_title("Taxa de churn por contrato")
axes[1].set_xlabel("taxa de churn")
for i, v in enumerate(by_contract.to_numpy()):
    axes[1].text(v, i, f" {v:.1%}", va="center", fontsize=9)
axes[1].set_xlim(0, by_contract.max() * 1.22)

# (c) tenure
axes[2].hist(
    [raw.loc[~churned, "tenure"], raw.loc[churned, "tenure"]],
    bins=24,
    label=["permaneceu", "saiu"],
    color=["#4C78A8", "#E45756"],
    density=True,
)
axes[2].set_title("Distribuicao de tenure")
axes[2].set_xlabel("meses de relacionamento")
axes[2].set_ylabel("densidade")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Taxa de churn geral: {churned.mean():.2%}")
print("\nTaxa de churn por contrato:")
for contract, rate in by_contract.items():
    print(f"  {contract:16s} {rate:6.2%}")

### Três observações que orientam o resto do trabalho

**1. O alvo é desbalanceado (26,5 % positivos).** Isso fixa a escolha de métrica:
Average Precision como critério primário, acurácia como auxiliar.

**2. O tipo de contrato separa fortemente.** Contrato mensal churna a uma taxa muito
superior à de contratos anuais e bienais. Isso é **associação**, não causa: o tipo de
contrato também é escolhido por clientes que já pretendem ficar pouco tempo. O modelo
pode usar o sinal; o relatório não pode chamá-lo de causa.

**3. `tenure` concentra saídas no início do relacionamento.** Clientes novos saem
muito mais. É o preditor que dominará a explicabilidade global — e faz sentido
substantivo, sem que isso o torne causal.

## 6. Divisão treino/teste — a decisão que protege todo o resto

Esta é a primeira decisão irreversível do projeto, e ela vem **antes** de qualquer
transformação.

| Parâmetro | Valor |
|---|---|
| `test_size` | `0.20` |
| `stratify` | pela coluna `Churn` |
| `random_state` | `42` |
| `shuffle` | `True` |

**Estratificado** porque a classe positiva é minoritária: uma partição aleatória
simples poderia deslocar a prevalência entre os lados e tornar as duas metades não
comparáveis. **Semente fixa** porque a partição precisa ser a mesma em toda execução,
em qualquer máquina.

O conjunto de teste (holdout) fica **intocado** até a seção 10. Ele não participa de
imputação, escala, codificação, seleção de features, comparação de modelos, busca de
hiperparâmetros, calibração ou escolha de limiar.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TEST_SIZE = 0.20


def identifiers_digest(identifiers) -> str:
    """SHA-256 do conjunto ordenado de identificadores.

    Ordenado antes de hashear: o digest descreve o *conjunto* de linhas, nao a ordem
    em que train_test_split as emitiu.
    """
    payload = "\n".join(sorted(str(value) for value in identifiers))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


training_pool, holdout = train_test_split(
    raw,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=raw[TARGET_COLUMN],
    shuffle=True,
)

# Digests congelados, de reports/split_manifest.json no repositorio.
EXPECTED_TRAINING_IDS_SHA256 = "a553196dd46b672f6344867707144fbf56a8abe14b338a225662c7208a1450dd"
EXPECTED_HOLDOUT_IDS_SHA256 = "1ad8aefb7e34776a78d76765d2465c630a41b3813b1d7d96d0d1b03d049776f6"

training_digest = identifiers_digest(training_pool[ID_COLUMN])
holdout_digest = identifiers_digest(holdout[ID_COLUMN])

print(
    f"Pool de treino : {len(training_pool):5d} linhas   "
    f"prevalencia {(training_pool[TARGET_COLUMN] == POSITIVE_LABEL).mean():.4f}"
)
print(
    f"Holdout        : {len(holdout):5d} linhas   "
    f"prevalencia {(holdout[TARGET_COLUMN] == POSITIVE_LABEL).mean():.4f}"
)

overlap = set(training_pool[ID_COLUMN]) & set(holdout[ID_COLUMN])
print(f"\nIntersecao entre as particoes: {len(overlap)} identificadores")

print("\nVerificacao contra a particao congelada do repositorio:")
print(
    f"  treino  {'OK' if training_digest == EXPECTED_TRAINING_IDS_SHA256 else 'DIVERGE'}"
    f"  {training_digest[:24]}..."
)
print(
    f"  holdout {'OK' if holdout_digest == EXPECTED_HOLDOUT_IDS_SHA256 else 'DIVERGE'}"
    f"  {holdout_digest[:24]}..."
)

assert len(training_pool) == 5634 and len(holdout) == 1409
assert not overlap
assert training_digest == EXPECTED_TRAINING_IDS_SHA256
assert holdout_digest == EXPECTED_HOLDOUT_IDS_SHA256
print("\nA particao reproduzida e, linha a linha, a particao congelada do projeto.")

> **O que acabou de ser verificado.** As duas partições contêm exatamente os mesmos
> clientes que o repositório congelou — não apenas a mesma quantidade. Como a
> partição é função pura dos bytes brutos mais a configuração, ela é **regenerada**
> em vez de armazenada, e o digest prova que a regeneração deu certo.

## 7. Pré-processamento sem vazamento

### O princípio

Vazamento é a falha crítica em projetos supervisionados: qualquer estatística
aprendida da partição de teste contamina a estimativa final e a torna otimista de um
jeito que nenhuma métrica revela.

A proteção estrutural é fazer todo passo com estado viver **dentro** de um
`Pipeline`, ajustado apenas nos dados de treino de cada fold:

| Passo | Tem estado? | O que aprende |
|---|---|---|
| `TotalChargesCleaner` | **não** | nada — a regra é determinística |
| `StandardScaler` | **sim** | média e desvio das 3 numéricas |
| `OneHotEncoder` | **sim** | os níveis das 16 categóricas |
| `LogisticRegression` | **sim** | os 46 coeficientes + intercepto |

Como os três passos com estado estão dentro do mesmo `Pipeline`, a validação cruzada
reajusta **todos** eles a cada fold. Não é disciplina do programador — é a estrutura
que impede o erro.

### Escala e codificação

`StandardScaler` nas numéricas porque a regressão logística com penalização L2 é
sensível à escala: sem padronizar, a penalização atinge desigualmente coeficientes de
features em unidades diferentes (meses, reais, reais acumulados).

`OneHotEncoder(handle_unknown="ignore")` nas categóricas. O `ignore` é uma decisão
operacional deliberada: uma categoria nunca vista em treino — um plano novo, um meio
de pagamento novo — vira um bloco de zeros e **contribui exatamente nada**, em vez de
derrubar a predição. O sistema continua respondendo, e o monitoramento conta o evento
(seção 14).

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


class TotalChargesCleaner(BaseEstimator, TransformerMixin):
    """Converte TotalCharges de texto para float64, com a regra do zero estrutural.

    Sem estado por construcao: `fit` registra apenas os nomes de coluna. Nenhuma
    estatistica e estimada dos dados, entao este passo nao pode vazar nada.
    """

    def __init__(self, column: str = "TotalCharges", tenure_column: str = "tenure") -> None:
        self.column = column
        self.tenure_column = tenure_column

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        frame = X.copy()
        text = frame[self.column].astype("string").str.strip()
        blank = text.isna() | (text == "")
        coerced = pd.to_numeric(text, errors="coerce")
        tenure = pd.to_numeric(frame[self.tenure_column], errors="coerce")

        unparseable = coerced.isna() & ~blank
        blank_with_history = blank & (tenure != 0)
        invalid = unparseable | blank_with_history
        if bool(invalid.any()):
            raise ValueError(
                f"{int(invalid.sum())} linha(s) com {self.column!r} invalido "
                f"({int(unparseable.sum())} nao numericas, "
                f"{int(blank_with_history.sum())} em branco com tenure > 0). "
                "Branco so e aceito onde o cliente nao completou um ciclo de faturamento."
            )

        frame[self.column] = coerced.where(~blank, 0.0).astype("float64")
        return frame

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_in_, dtype=object)


def build_pipeline() -> Pipeline:
    """O pipeline congelado do projeto: limpeza -> escala/codificacao -> logistica."""
    return Pipeline(
        [
            ("clean", TotalChargesCleaner()),
            (
                "preprocessor",
                ColumnTransformer(
                    [
                        ("numeric", StandardScaler(), NUMERIC_FEATURES),
                        (
                            "categorical",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False, dtype=np.float64
                            ),
                            CATEGORICAL_FEATURES,
                        ),
                    ]
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=1.0,
                    l1_ratio=0.0,
                    solver="lbfgs",
                    class_weight=None,
                    max_iter=100,
                ),
            ),
        ]
    )


X_train = training_pool.drop(columns=[ID_COLUMN, TARGET_COLUMN])
y_train = (training_pool[TARGET_COLUMN] == POSITIVE_LABEL).astype(int)
X_holdout = holdout.drop(columns=[ID_COLUMN, TARGET_COLUMN])
y_holdout = (holdout[TARGET_COLUMN] == POSITIVE_LABEL).astype(int)

probe = build_pipeline().fit(X_train, y_train)
n_transformed = (
    probe.named_steps["preprocessor"]
    .transform(probe.named_steps["clean"].transform(X_train))
    .shape[1]
)

print(f"Features de entrada     : {X_train.shape[1]}")
print(f"Colunas apos o encoding : {n_transformed}")
print(f"  {len(NUMERIC_FEATURES)} numericas padronizadas")
print(
    f"  {n_transformed - len(NUMERIC_FEATURES)} indicadores one-hot "
    f"de {len(CATEGORICAL_FEATURES)} features categoricas"
)
print(
    f"\nConvergiu em {probe.named_steps['classifier'].n_iter_[0]} iteracoes "
    f"(limite {probe.named_steps['classifier'].max_iter})"
)

## 8. Protocolo experimental e baselines

### O protocolo

Toda comparação deste trabalho usa **a mesma partição de validação cruzada**:
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`, sobre o pool de treino,
compartilhada entre todos os modelos. Isso torna as diferenças **pareadas**: os
modelos são avaliados nas mesmas linhas, nos mesmos folds.

O holdout **não aparece** em nenhuma célula desta seção.

### Por que baselines vêm primeiro

Um baseline não é formalidade. Ele responde "quanto disso é o modelo, e quanto é a
estrutura do problema?" Sem `DummyClassifier`, a acurácia de 80 % da logística
parece boa; com ele, vê-se que 73,5 % vinham de graça.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
DIAGNOSTIC_THRESHOLD = 0.5  # referencia apenas; o limiar real e escolhido na secao 9


def cross_validate_pipeline(factory, X, y, cv=CV, threshold=DIAGNOSTIC_THRESHOLD):
    """Avalia um pipeline nos folds compartilhados e devolve metricas + probs OOF.

    O pipeline inteiro e reajustado dentro de cada fold, entao escala e categorias
    sao aprendidas somente do treino daquele fold.
    """
    out_of_fold = np.zeros(len(y), dtype=float)
    rows = []
    for fold, (train_index, valid_index) in enumerate(cv.split(X, y), start=1):
        model = factory().fit(X.iloc[train_index], y.iloc[train_index])
        probability = model.predict_proba(X.iloc[valid_index])[:, 1]
        out_of_fold[valid_index] = probability
        predicted = (probability >= threshold).astype(int)
        truth = y.iloc[valid_index]
        rows.append(
            {
                "fold": fold,
                "average_precision": average_precision_score(truth, probability),
                "roc_auc": roc_auc_score(truth, probability),
                "precision": precision_score(truth, predicted, zero_division=0),
                "recall": recall_score(truth, predicted, zero_division=0),
                "f1": f1_score(truth, predicted, zero_division=0),
                "accuracy": accuracy_score(truth, predicted),
            }
        )
    folds = pd.DataFrame(rows).set_index("fold")
    return folds, out_of_fold


def summarise(folds: pd.DataFrame) -> pd.Series:
    """Media e desvio-padrao amostral (ddof=1) por metrica, como o repositorio grava."""
    return pd.concat([folds.mean().rename("mean"), folds.std(ddof=1).rename("std")], axis=1)

In [ ]:
baselines = {}

for name, factory in {
    "B0 maioria (DummyClassifier)": lambda: DummyClassifier(strategy="most_frequent"),
    "B1 aleatorio estratificado": lambda: DummyClassifier(
        strategy="stratified", random_state=RANDOM_SEED
    ),
    "B2 regressao logistica": build_pipeline,
}.items():
    folds, oof = cross_validate_pipeline(factory, X_train, y_train)
    baselines[name] = {"folds": folds, "oof": oof, "summary": summarise(folds)}

baseline_table = pd.DataFrame(
    {name: result["summary"]["mean"] for name, result in baselines.items()}
).T[["average_precision", "roc_auc", "f1", "recall", "precision", "accuracy"]]

print("Baselines - validacao cruzada 5-fold no pool de treino (limiar diagnostico 0,5)\n")
print(baseline_table.round(4).to_string())

print(f"\nAP sem informacao (= prevalencia) : {y_train.mean():.4f}")
print("ROC-AUC sem informacao            : 0.5000")

### Leitura dos baselines

O `DummyClassifier` de classe majoritária alcança **~73,5 % de acurácia** e
**F1 = 0** — ele nunca identifica um cliente em risco. É exatamente a demonstração de
que acurácia, isolada, não mede utilidade neste problema.

A regressão logística sobe o Average Precision de **~0,265** (a prevalência, que é o
AP de um classificador sem informação) para **~0,66**. Esse é o ganho real: a
capacidade de **ordenar** clientes por risco.

Repare que, com o limiar diagnóstico de 0,5, o recall da logística fica em torno de
0,54 — ela deixa passar quase metade dos clientes que saem. Isso não é defeito do
modelo: é consequência do limiar. A seção 9 trata disso.

## 9. Comparação de modelos

### As famílias consideradas, e por quê

Três famílias, cada uma com uma hipótese explícita — não uma varredura de dezenas de
algoritmos:

| | Família | Hipótese |
|---|---|---|
| **M0** | Regressão logística | log-odds aditivo nas features codificadas. É a **referência**, não um candidato. |
| **M1** | Random forest | árvores profundas decorrelacionadas capturam interações e efeitos não monótonos de tenure/cobranças que uma inclinação aditiva não expressa. |
| **M2** | Histogram gradient boosting | árvores rasas ajustadas sequencialmente ao gradiente atacam justamente os erros que um modelo linear deixa. |

Cada candidato tem de **justificar sua substituição** da referência. O padrão, quando
não justifica, é ficar com o modelo mais simples — não adotar o de maior pontuação.

In [ ]:
comparison = {"M0 regressao logistica": baselines["B2 regressao logistica"]}

for name, factory in {
    "M1 random forest": lambda: Pipeline(
        [
            ("clean", TotalChargesCleaner()),
            (
                "preprocessor",
                ColumnTransformer(
                    [
                        ("numeric", StandardScaler(), NUMERIC_FEATURES),
                        (
                            "categorical",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False, dtype=np.float64
                            ),
                            CATEGORICAL_FEATURES,
                        ),
                    ]
                ),
            ),
            (
                "classifier",
                RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=1),
            ),
        ]
    ),
    "M2 hist gradient boosting": lambda: Pipeline(
        [
            ("clean", TotalChargesCleaner()),
            (
                "preprocessor",
                ColumnTransformer(
                    [
                        ("numeric", StandardScaler(), NUMERIC_FEATURES),
                        (
                            "categorical",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False, dtype=np.float64
                            ),
                            CATEGORICAL_FEATURES,
                        ),
                    ]
                ),
            ),
            (
                "classifier",
                HistGradientBoostingClassifier(random_state=RANDOM_SEED, early_stopping=False),
            ),
        ]
    ),
}.items():
    folds, oof = cross_validate_pipeline(factory, X_train, y_train)
    comparison[name] = {"folds": folds, "oof": oof, "summary": summarise(folds)}

rows = []
for name, result in comparison.items():
    summary = result["summary"]
    rows.append(
        {
            "modelo": name,
            "AP (media)": summary.loc["average_precision", "mean"],
            "AP (dp)": summary.loc["average_precision", "std"],
            "ROC-AUC (media)": summary.loc["roc_auc", "mean"],
            "ROC-AUC (dp)": summary.loc["roc_auc", "std"],
        }
    )
comparison_table = pd.DataFrame(rows).set_index("modelo")

print("Comparacao de modelos - AP como criterio primario\n")
print(comparison_table.round(4).to_string())

# Paridade com reports/experiments/model_comparison_results.json (repositorio).
FROZEN_COMPARISON_AP = {
    "M0 regressao logistica": 0.661493,
    "M1 random forest": 0.606181,
    "M2 hist gradient boosting": 0.647137,
}
print("\nParidade com o artefato versionado (AP media):")
for name, frozen in FROZEN_COMPARISON_AP.items():
    reproduced = comparison_table.loc[name, "AP (media)"]
    print(
        f"  {name:28s} reproduzido {reproduced:.6f}  artefato {frozen:.6f}  "
        f"delta {reproduced - frozen:+.2e}"
    )

# Deltas pareados por fold contra a referencia M0.
reference_folds = comparison["M0 regressao logistica"]["folds"]["average_precision"]
print("\nDeltas pareados de AP contra M0 (por fold):")
for name in ("M1 random forest", "M2 hist gradient boosting"):
    delta = comparison[name]["folds"]["average_precision"] - reference_folds
    print(f"  {name:28s} media {delta.mean():+.5f}   folds melhores: {int((delta > 0).sum())}/5")

### O resultado, e a decisão que ele sustenta

Nenhuma das duas famílias baseadas em árvores supera a regressão logística em Average
Precision sob este protocolo. A random forest fica claramente abaixo; o boosting
chega perto, mas abaixo.

Isso é menos surpreendente do que parece. Depois do one-hot, a maior parte do sinal
deste problema é aproximadamente aditiva em log-odds: tenure baixo, contrato mensal,
fibra, cheque eletrônico. Não há uma estrutura de interação rica que justifique a
capacidade extra — e capacidade sem sinal para explicar vira variância.

**Decisão: a regressão logística permanece.** Ela vence na métrica primária, é a mais
simples e é a única cuja decisão pode ser decomposta exatamente (seção 12).

## 10. Engenharia de atributos

### O que foi feito — e o que foi decidido

Um ponto que precisa ser dito com precisão, porque é fácil ler errado:
**engenharia de atributos foi executada.** Cinco grupos de features foram formulados
a partir de hipóteses substantivas e avaliados sob o protocolo congelado. O que **não**
aconteceu foi a *adoção* de qualquer um deles no modelo final.

Isso não é ausência de engenharia de atributos. É engenharia de atributos com um
critério de adoção: **uma feature só entra se demonstrar benefício suficiente para
justificar a complexidade que adiciona.**

| | Grupo | Hipótese | Resultado (Δ AP pareado, 5 folds) |
|---|---|---|---|
| **E1** | `protective_services` | contagem de serviços de proteção carrega sinal que os 4 dummies não expressam como uma inclinação | −0,0000, melhora em 2/5 → **NÃO SUSTENTADO** |
| **E2** | `automatic_payment` | o eixo manual/automático estima o sinal de pagamento mais estavelmente que 4 dummies | +0,0001, melhora em 3/5 → **INCONCLUSIVO** |
| **E3** | `contract_tenure` | a interação contrato × tenure captura risco que nenhum dos dois sozinho expressa | +0,0017, melhora em 4/5 → **PROMISSOR** |
| **E4** | `historical_average_charge` | cobrança média histórica separa o cliente caro do cliente antigo | −0,0015, melhora em 1/5 → **NÃO SUSTENTADO** |
| **E5** | `charge_intensity` | cobrança relativa à mediana da faixa capta desalinhamento de preço | −0,0005, melhora em 2/5 → **NÃO SUSTENTADO** |

### Por que E1 e E4/E5 falharam

E1 é instrutivo: a contagem de serviços protetores é uma **soma determinística de
indicadores que o modelo já recebe one-hot**. Ela não acrescenta informação — apenas
uma parametrização mais grosseira de informação já presente. O resultado nulo é o
resultado correto, e tê-lo medido é o que permite afirmar isso.

### Por que E3, mesmo promissor, não foi adotado

E3 melhorou em 4 de 5 folds, com ganho médio de **+0,0017 de AP** — direção
consistente, magnitude pequena. O custo é concreto: 2 features a mais, 48 colunas em
vez de 46, e uma interação a explicar em cada explicação local.

O ganho não paga esse custo. E3 foi mantido como **análise de sensibilidade** — foi
carregado para a comparação de modelos como conjunto alternativo de features, para
verificar que a escolha do modelo não dependia da representação — mas **não entrou no
pipeline final**.

> **Nenhum teste de significância foi executado.** Cinco folds não sustentam um, e as
> classificações PROMISSOR / INCONCLUSIVO / NÃO SUSTENTADO são heurísticas de
> engenharia sobre direção e consistência, não inferência estatística.

O modelo final usa, portanto, as **19 features originais**.

## 11. Tuning e política de calibração

Estas duas etapas foram executadas no repositório com validação cruzada aninhada e
são **narradas** aqui: reproduzi-las exigiria dezenas de minutos de busca em grade e
não mudaria nenhuma conclusão. Os números vêm de
`reports/experiments/tuning_results.json` e `calibration_results.json`.

### Tuning — validação cruzada aninhada

O erro comum em tuning é medir o desempenho do modelo **nos mesmos dados que
escolheram seus hiperparâmetros**, o que produz uma estimativa otimista. A proteção é
validação cruzada aninhada: um laço externo de 5 folds estima o desempenho do
*procedimento*, e dentro de cada fold de treino um laço interno de 4 folds escolhe os
hiperparâmetros. A escolha nunca vê as linhas em que é avaliada.

| | Procedimento | AP externo (média ± dp) | Δ pareado vs T0 | Folds melhores |
|---|---|---|---|---|
| **T0** | Logística congelada (C = 1,0, sem tuning) | 0,661493 ± 0,0217 | — | — |
| **T1** | Logística com C buscado | 0,661298 ± 0,0211 | −0,00020 | 2/5 |
| **T2** | Histogram gradient boosting ajustado | 0,665335 ± 0,0271 | **+0,00384** | 3/5 |

A regra de elegibilidade foi **fixada antes de ver os resultados**: substituir a
referência exige delta médio de AP positivo **e** melhora em pelo menos 4 dos 5 folds
externos.

**T2 tem o maior AP médio e mesmo assim não foi adotado.** Ganha em 3 folds e perde em
2 — a vantagem média vem de um ganho grande em poucos folds, não de superioridade
consistente. Adotá-lo seria escolher pelo maior número em vez de por evidência de que
ele é de fato melhor. O padrão pré-registrado se aplica: **mantém-se o modelo mais
simples**.

### Calibração — avaliada, não adotada

Igualmente importante dizer com precisão: **calibração foi experimentada.** Dois
métodos foram testados contra o modelo não calibrado, com Brier score como métrica de
decisão primária e log loss como secundária.

| | Método | Brier | Log loss | Δ AP | Situação |
|---|---|---|---|---|---|
| **C0** | Não calibrado | 0,135064 | 0,416680 | — | referência |
| **C1** | Sigmoid (Platt) | 0,135081 | 0,416597 | +0,00000 | melhora Brier em **1/5** folds |
| **C2** | Isotônica | 0,135388 | 0,428612 | −0,01513 | piora AP em **5/5** folds |

Nenhum método cumpriu o protocolo de adoção. O sigmoid praticamente não muda nada — o
que faz sentido: uma regressão logística já produz probabilidades razoavelmente
calibradas por construção. A isotônica piora tudo, sinal clássico de sobreajuste com
essa quantidade de dados.

**Resultado: `calibration_policy = NONE`.** Nenhuma camada de calibração foi adotada.

A consequência precisa ser dita: os scores do modelo são **posições de ordenação**, não
frequências calibradas. Uma probabilidade de 0,42 não deve ser lida como "42 % destes
clientes sairão".

## 12. O limiar de decisão

### Por que 0,5 é uma convenção, não uma escolha

`predict()` usa 0,5 por padrão. Esse número não vem do problema — vem da
implementação. Em um problema desbalanceado com custos assimétricos, usá-lo sem
examinar é aceitar uma decisão que ninguém tomou.

O modelo produz um score contínuo. O limiar converte score em ação, e é onde a
assimetria entre erros entra:

* um **falso negativo** é um cliente em risco que a operação nunca contatou — a
  chance de retenção foi perdida;
* um **falso positivo** é um cliente estável que recebeu contato — custa esforço e
  eventualmente um desconto desnecessário.

### Como o limiar foi escolhido — sem tocar no holdout

A política é **maximização de F1**, aplicada às probabilidades *out-of-fold* do pool
de treino. A escolha do procedimento foi validada de forma aninhada: em cada fold
externo, o limiar era escolhido apenas nos dados de treino daquele fold e avaliado nos
de validação, que não participaram de escolhê-lo.

Resultado dessa validação: **melhora de F1 em 5 dos 5 folds externos**, com delta
médio de **+0,0409**. O procedimento bate o padrão consistentemente, então é aplicado
uma última vez ao pool de treino inteiro para fixar o limiar que vai ao holdout.

> **F1 é uma escolha declarada, não uma verdade.** Ele equilibra precisão e recall
> tratando-os como igualmente importantes. Isso é uma **suposição**, adotada porque
> este trabalho **não dispõe de custos reais de campanha de retenção nem de valor de
> cliente**. Nenhum número de negócio foi inventado para justificá-la. Com custos
> reais, o limiar correto seria o que minimiza custo esperado, e quase certamente não
> seria este.

In [ ]:
# Probabilidades out-of-fold da logistica, ja calculadas na secao 8.
oof_probability = comparison["M0 regressao logistica"]["oof"]


def maximise_f1(y_true, probability):
    """Devolve o limiar que maximiza F1 sobre os scores observados.

    Varre os valores de score distintos: o F1 so muda quando o limiar cruza um score,
    entao esses sao os unicos candidatos que importam.
    """
    best_threshold, best_f1 = 0.5, -1.0
    for candidate in np.unique(probability):
        score = f1_score(y_true, (probability >= candidate).astype(int), zero_division=0)
        if score > best_f1:
            best_f1, best_threshold = score, float(candidate)
    return best_threshold, best_f1


selected_threshold, selected_f1 = maximise_f1(y_train.to_numpy(), oof_probability)

FROZEN_THRESHOLD = 0.3272694566222328  # de reports/decision_policy.json
print(f"Limiar reproduzido : {selected_threshold!r}")
print(f"Limiar congelado   : {FROZEN_THRESHOLD!r}")
print(f"Identicos          : {selected_threshold == FROZEN_THRESHOLD}")

In [ ]:
# O trade-off, medido nas mesmas probabilidades out-of-fold.
grid = np.linspace(0.05, 0.95, 181)
curve = pd.DataFrame(
    {
        "threshold": grid,
        "precision": [
            precision_score(y_train, (oof_probability >= t).astype(int), zero_division=0)
            for t in grid
        ],
        "recall": [
            recall_score(y_train, (oof_probability >= t).astype(int), zero_division=0) for t in grid
        ],
        "f1": [
            f1_score(y_train, (oof_probability >= t).astype(int), zero_division=0) for t in grid
        ],
    }
)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))
for column, colour in (("precision", "#4C78A8"), ("recall", "#E45756"), ("f1", "#54A24B")):
    axes[0].plot(curve["threshold"], curve[column], label=column, color=colour)
axes[0].axvline(
    FROZEN_THRESHOLD, ls="--", c="#333", lw=1.2, label=f"congelado {FROZEN_THRESHOLD:.4f}"
)
axes[0].axvline(0.5, ls=":", c="#888", lw=1.2, label="padrao 0,5")
axes[0].set_xlabel("limiar")
axes[0].set_ylabel("metrica")
axes[0].set_title("Trade-off precisao / recall (out-of-fold, treino)")
axes[0].legend(fontsize=8)

comparison_rows = []
for label, t in (("padrao 0,5", 0.5), (f"congelado {FROZEN_THRESHOLD:.4f}", FROZEN_THRESHOLD)):
    predicted = (oof_probability >= t).astype(int)
    comparison_rows.append(
        {
            "limiar": label,
            "precisao": precision_score(y_train, predicted, zero_division=0),
            "recall": recall_score(y_train, predicted, zero_division=0),
            "f1": f1_score(y_train, predicted, zero_division=0),
            "acuracia": accuracy_score(y_train, predicted),
            "taxa de positivos": predicted.mean(),
        }
    )
threshold_comparison = pd.DataFrame(comparison_rows).set_index("limiar")

axes[1].axis("off")
axes[1].table(
    cellText=threshold_comparison.round(4).to_numpy(),
    rowLabels=threshold_comparison.index,
    colLabels=threshold_comparison.columns,
    loc="center",
    cellLoc="center",
).scale(1.0, 1.6)
axes[1].set_title("Padrao contra limiar congelado")
plt.tight_layout()
plt.show()

print(threshold_comparison.round(4).to_string())

### O que baixar o limiar de 0,50 para 0,327 realmente faz

| | Padrão 0,5 | Congelado 0,327 |
|---|---|---|
| Recall | ~0,54 | **~0,74** |
| Precisão | ~0,65 | **~0,56** |
| Clientes sinalizados | ~22 % | **~35 %** |

O modelo passa a encontrar cerca de **três quartos** dos clientes que sairão, em vez
de pouco mais da metade. O preço é que **quase metade** dos sinalizados não iria sair,
e a operação precisa de capacidade para contatar 35 % da carteira em vez de 22 %.

**Isto não é "o limiar ótimo para o negócio".** É o limiar que a política declarada
(maximização de F1) seleciona com os dados disponíveis. A decisão certa depende do
custo de uma campanha de retenção, do valor de um cliente retido e da capacidade
operacional — três números que este trabalho não possui e não vai inventar.

> Os números acima são *out-of-fold no pool de treino*, e são as métricas que
> **selecionaram** o limiar. Por isso são otimistas e **não** são desempenho final.
> Desempenho final só existe depois da próxima seção.

## 13. Avaliação final no holdout

### As regras desta seção

O holdout é aberto **uma vez**. Toda decisão — modelo, features, hiperparâmetros,
política de calibração, limiar — já está congelada. Nada aqui pode mudar nenhuma
delas.

```
evaluations_performed        = 1
selection_after_holdout      = False
```

Isso não é cerimônia. Um conjunto de teste só fornece uma estimativa não enviesada
enquanto **nenhuma decisão** tiver sido tomada olhando para ele. Se olhássemos o
resultado e voltássemos para ajustar qualquer coisa, a estimativa deixaria de valer —
e nenhuma métrica denunciaria isso.

O modelo final é ajustado no pool de treino **inteiro** (5634 linhas), com a
configuração congelada, e aplicado ao holdout **uma vez**.

In [ ]:
final_model = build_pipeline().fit(X_train, y_train)

holdout_probability = final_model.predict_proba(X_holdout)[:, 1]
holdout_prediction = (holdout_probability >= FROZEN_THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(y_holdout, holdout_prediction).ravel()

holdout_metrics = {
    "average_precision": average_precision_score(y_holdout, holdout_probability),
    "roc_auc": roc_auc_score(y_holdout, holdout_probability),
    "recall": recall_score(y_holdout, holdout_prediction),
    "precision": precision_score(y_holdout, holdout_prediction),
    "f1": f1_score(y_holdout, holdout_prediction),
    "accuracy": accuracy_score(y_holdout, holdout_prediction),
}

# Valores congelados em reports/experiments/holdout_results.json (Fase 9D).
FROZEN_HOLDOUT = {
    "average_precision": 0.633702,
    "roc_auc": 0.842034,
    "recall": 0.721925,
    "precision": 0.537849,
    "f1": 0.616438,
    "accuracy": 0.761533,
}
FROZEN_CONFUSION = {"tn": 803, "fp": 232, "fn": 104, "tp": 270}

print(
    f"Holdout: {len(y_holdout)} clientes  "
    f"({int(y_holdout.sum())} sairam, {int((1 - y_holdout).sum())} permaneceram)\n"
)

print(f"{'metrica':>18s} {'reproduzido':>12s} {'congelado':>11s} {'diferenca':>11s}")
print("-" * 56)
for key, value in holdout_metrics.items():
    frozen = FROZEN_HOLDOUT[key]
    print(f"{key:>18s} {value:12.6f} {frozen:11.6f} {value - frozen:+11.2e}")

observed_confusion = {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
print(f"\nMatriz de confusao reproduzida : {observed_confusion}")
print(f"Matriz de confusao congelada   : {FROZEN_CONFUSION}")
print(f"Identicas                      : {observed_confusion == FROZEN_CONFUSION}")

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2))

# (a) Precision-Recall
precision_curve, recall_curve, _ = precision_recall_curve(y_holdout, holdout_probability)
axes[0].plot(recall_curve, precision_curve, color="#4C78A8", lw=2)
axes[0].axhline(
    float(y_holdout.mean()),
    ls="--",
    c="#888",
    lw=1.2,
    label=f"sem informacao ({y_holdout.mean():.3f})",
)
axes[0].scatter(
    [holdout_metrics["recall"]],
    [holdout_metrics["precision"]],
    color="#E45756",
    zorder=5,
    s=55,
    label="ponto de operacao congelado",
)
axes[0].set_xlabel("recall")
axes[0].set_ylabel("precisao")
axes[0].set_title(f"Precision-Recall (AP = {holdout_metrics['average_precision']:.3f})")
axes[0].legend(fontsize=8)

# (b) ROC
fpr, tpr, _ = roc_curve(y_holdout, holdout_probability)
axes[1].plot(fpr, tpr, color="#4C78A8", lw=2)
axes[1].plot([0, 1], [0, 1], ls="--", c="#888", lw=1.2, label="sem informacao")
axes[1].set_xlabel("taxa de falsos positivos")
axes[1].set_ylabel("taxa de verdadeiros positivos")
axes[1].set_title(f"ROC (AUC = {holdout_metrics['roc_auc']:.3f})")
axes[1].legend(fontsize=8)

# (c) Matriz de confusao
matrix = np.array([[tn, fp], [fn, tp]])
axes[2].imshow(matrix, cmap="Blues")
axes[2].set_xticks([0, 1], ["previsto: fica", "previsto: sai"])
axes[2].set_yticks([0, 1], ["real: ficou", "real: saiu"])
for i in range(2):
    for j in range(2):
        axes[2].text(
            j,
            i,
            f"{matrix[i, j]}",
            ha="center",
            va="center",
            fontsize=15,
            color="white" if matrix[i, j] > matrix.max() / 2 else "black",
        )
axes[2].set_title("Matriz de confusao (limiar congelado)")
axes[2].grid(False)

plt.tight_layout()
plt.show()

## 14. Análise das métricas

### Os números, e o que cada um significa

| Métrica | Valor | IC 95 % | Leitura |
|---|---|---|---|
| **Average Precision** | **0,634** | [0,578 – 0,686] | qualidade de ordenação; sem informação seria 0,265 |
| **ROC-AUC** | **0,842** | [0,818 – 0,863] | separação entre as classes; sem informação seria 0,500 |
| **Recall** | **0,722** | [0,676 – 0,765] | encontra ~72 % de quem realmente saiu |
| **Precisão** | **0,538** | [0,494 – 0,579] | ~54 % dos sinalizados de fato saíram |
| **F1** | **0,616** | [0,576 – 0,653] | equilíbrio no ponto de operação congelado |
| Acurácia | 0,762 | — | **auxiliar** — a classe majoritária já dá 0,735 |

Os intervalos vêm de bootstrap percentil com 2000 reamostragens do holdout
(semente 42). O modelo **não** é reajustado a cada reamostragem, e o limiar **não** é
reescolhido: o intervalo descreve a incerteza da estimativa sobre esta população, não
a variabilidade de todo o procedimento.

### Cuidados de linguagem que este relatório mantém

**ROC-AUC de 0,842 não é "84 % de acerto".** É a probabilidade de o modelo ordenar um
cliente que saiu acima de um que ficou, tomados ao acaso. Confundir as duas coisas é o
erro de leitura mais comum em relatórios de churn.

**Acurácia de 0,762 é quase irrelevante aqui.** A classe majoritária entrega 0,735 de
graça. O ganho de 2,7 pontos percentuais descreve mal um modelo que triplicou a
capacidade de encontrar clientes em risco em relação ao acaso.

**Average Precision de 0,634 contra 0,265 sem informação** é a comparação que
realmente importa: em uma população onde 26,5 % saem, o modelo ordena de forma que a
precisão média ao longo de toda a curva seja 2,4 vezes a de referência.

### Generalização: desenvolvimento contra holdout

| Métrica | Desenvolvimento (CV) | Holdout | Diferença |
|---|---|---|---|
| Average Precision | 0,661493 | 0,633702 | **−0,027791** |
| ROC-AUC | 0,846149 | 0,842034 | **−0,004115** |

A queda é **pequena e na direção esperada**. Estimativas de validação cruzada no pool
que participou de todas as escolhas tendem a ser levemente otimistas; ver o holdout um
pouco abaixo é o comportamento normal de um procedimento que não sobreajustou.

> Isto é uma **descrição**, não um teste. Nenhuma hipótese foi formulada sobre essa
> diferença, nenhum p-valor foi calculado, e nenhum limite de aprovação foi definido
> antes de olhá-la. Ela é reportada porque é informativa, não porque foi validada.

## 15. A limitação que não pode ser omitida — exposição do analista

O holdout foi protegido corretamente **a partir do momento da divisão**: nenhum
ajuste, nenhuma seleção, nenhuma busca de hiperparâmetro, nenhuma escolha de limiar o
tocou.

Mas a análise exploratória inicial deste projeto foi conduzida **sobre as 7043 linhas
completas**, antes de o holdout ser formalmente separado. As hipóteses que orientaram
o trabalho — que contrato importa, que tenure domina, que fibra se associa a mais
churn — foram formadas olhando dados que incluíam as linhas que depois viraram o
conjunto de teste.

**Consequência honesta:**

> A estimativa final pode carregar um viés otimista que não é quantificável aqui.

Não há como medir esse viés com os dados existentes: seria preciso um segundo
conjunto de teste, coletado depois e nunca observado. Ele não existe.

O que é possível afirmar com precisão:

* **vazamento algorítmico não ocorreu** — nenhuma estatística foi ajustada no holdout,
  e o pipeline garante isso estruturalmente;
* **vazamento de conhecimento do analista ocorreu**, em grau desconhecido;
* o efeito é provavelmente pequeno — as hipóteses exploratórias eram sobre estrutura
  geral do problema, não sobre linhas individuais — mas "provavelmente pequeno" é uma
  avaliação, não uma medida.

Registrar isso custa credibilidade aparente e a devolve em confiabilidade. Um
relatório que omite essa limitação afirma mais precisão do que possui.

## 16. Explicabilidade global

### Por que não SHAP

SHAP, LIME e importância por permutação existem para sondar modelos cuja superfície de
resposta é **desconhecida**. Este modelo é conhecido em forma fechada:

$$\text{logit}(P) = \beta_0 + \sum_{f=1}^{19} \text{contribuição}_f, \qquad P = \sigma(\text{logit})$$

Cada contribuição é o termo aditivo daquela feature bruta no preditor linear — a soma
dos coeficientes das colunas one-hot que ela ocupa, vezes os valores transformados.
A decomposição é **exata**, não uma aproximação.

Usar um estimador por amostragem aqui adicionaria uma dependência, uma semente
aleatória e um erro de aproximação em troca de uma resposta **pior** a uma pergunta já
respondida exatamente. Não usar SHAP não é uma limitação deste trabalho — é a escolha
correta para esta classe de modelo.

### O ranking, e o que ele não é

O ranking abaixo ordena as features pela **dispersão empírica da contribuição** no
pool de treino: o desvio-padrão do termo que aquela feature adiciona ao logit, ao
longo dos 5634 clientes. Ele responde "quais features fazem o score variar mais nesta
população".

Ele **não** é:

* importância causal — nada aqui identifica um efeito;
* importância de negócio — nada aqui pondera valor de cliente;
* uma propriedade estável do problema — depende dos coeficientes ajustados, do
  pré-processamento ajustado, da composição do pool de treino e da correlação entre as
  features.

In [ ]:
classifier = final_model.named_steps["classifier"]
preprocessor = final_model.named_steps["preprocessor"]

intercept = float(classifier.intercept_[0])
coefficients = classifier.coef_[0]
transformed_names = list(preprocessor.get_feature_names_out())


# Mapeia cada coluna transformada de volta para sua feature bruta.
def owning_feature(column_name: str) -> str:
    body = column_name.split("__", 1)[1]
    if body in NUMERIC_FEATURES:
        return body
    for feature in sorted(CATEGORICAL_FEATURES, key=len, reverse=True):
        if body.startswith(feature + "_"):
            return feature
    raise ValueError(f"Coluna nao mapeada: {column_name}")


column_owner = [owning_feature(name) for name in transformed_names]
raw_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES

cleaned_train = final_model.named_steps["clean"].transform(X_train)
design_matrix = preprocessor.transform(cleaned_train)
term_matrix = design_matrix * coefficients  # (n_linhas, 46)


# Soma os termos de cada feature bruta -> contribuicao por feature, por linha.
def columns_of(feature):
    """Indices of the transformed columns that belong to one raw feature."""
    return [i for i, owner in enumerate(column_owner) if owner == feature]


contributions = pd.DataFrame(
    {feature: term_matrix[:, columns_of(feature)].sum(axis=1) for feature in raw_features}
)

# Verificacao: intercepto + soma das 19 contribuicoes tem de reproduzir o logit.
reconstructed_logit = intercept + contributions.to_numpy().sum(axis=1)
reference_logit = final_model.decision_function(X_train)
max_error = float(np.max(np.abs(reconstructed_logit - reference_logit)))

print(f"Intercepto : {intercept:.6f}")
print(f"Colunas transformadas : {len(transformed_names)}  ->  {len(raw_features)} features brutas")
print(f"\nErro maximo de reconstrucao do logit: {max_error:.3e}  (tolerancia 1e-12)")
print(f"A identidade se fecha: {max_error <= 1e-12}")

In [ ]:
dispersion = pd.DataFrame(
    {
        "dispersao (dp em log-odds)": contributions.std(ddof=0),
        "IQR": contributions.quantile(0.75) - contributions.quantile(0.25),
        "tipo": [
            "numerica" if f in NUMERIC_FEATURES else "categorica" for f in contributions.columns
        ],
    }
).sort_values("dispersao (dp em log-odds)", ascending=False)

print("Dispersao empirica da contribuicao no pool de treino (5634 clientes)\n")
print(dispersion.round(4).to_string())

fig, ax = plt.subplots(figsize=(8.4, 5.6))
ordered = dispersion.iloc[::-1]
colours = ["#4C78A8" if t == "numerica" else "#72B7B2" for t in ordered["tipo"]]
ax.barh(ordered.index, ordered["dispersao (dp em log-odds)"], color=colours)
ax.set_xlabel("desvio-padrao da contribuicao (log-odds)")
ax.set_title("Dispersao da contribuicao por feature bruta - pool de treino")
plt.tight_layout()
plt.show()

### O que o ranking mostra

**`tenure` domina** com folga — dispersão cerca de duas vezes a da segunda colocada.
Faz sentido substantivo: é a única feature que varia continuamente de 0 a 72 e cujo
coeficiente é grande, então move o score ao longo de toda a carteira.

**`MonthlyCharges`, `InternetService`, `Contract` e `TotalCharges`** formam o segundo
bloco. Note que `InternetService` e `Contract` aparecem alto apesar de terem apenas
três níveis cada: um punhado de níveis com coeficientes distantes produz muita
dispersão.

**`gender`, `Partner` e `PhoneService` são praticamente inertes** — dispersão abaixo
de 0,012 em log-odds. O modelo aprendeu que essas features quase não movem o score.

> **Uma advertência de parametrização.** A codificação one-hot é redundante com o
> intercepto: existem infinitas combinações (intercepto, coeficientes) que produzem
> exatamente as mesmas probabilidades. O coeficiente de um *nível individual* é
> portanto uma propriedade do ajuste, não uma quantidade estável. A **contribuição
> agregada por feature bruta** — o que este ranking usa — é bem mais robusta, mas ainda
> depende da representação escolhida.

## 17. Dependências estruturais — sete colunas, um único fato

Este é um ponto que a maior parte dos trabalhos sobre este dataset ignora, e que muda
a leitura de qualquer explicação.

Quando um cliente não tem internet, **seis colunas mudam juntas, mecanicamente**:

```
InternetService = No   =>   OnlineSecurity   = "No internet service"
                            OnlineBackup     = "No internet service"
                            DeviceProtection = "No internet service"
                            TechSupport      = "No internet service"
                            StreamingTV      = "No internet service"
                            StreamingMovies  = "No internet service"
```

E, de forma análoga:

```
PhoneService = No      =>   MultipleLines    = "No phone service"
```

Essas não são sete evidências independentes sobre o cliente. São **um fato**
— "este cliente não tem internet" — codificado em sete colunas.

**Por que isso importa para a explicabilidade.** Uma explicação local que liste sete
linhas separadas, cada uma com sua contribuição, sugere sete razões distintas quando
existe uma. O leitor conclui que o modelo considerou muitos aspectos, quando considerou
um aspecto sete vezes.

**O tratamento adotado:** as features acopladas são apresentadas como **um bloco
agregado**, cujo valor é a **soma exata** das contribuições de seus membros. Somar é
legítimo precisamente porque as contribuições são termos aditivos do mesmo logit — o
bloco continua particionando o logit exatamente.

In [ ]:
STRUCTURAL_BLOCKS = {
    "Sem internet": {
        "trigger": ("InternetService", "No"),
        "dependents": [
            "OnlineSecurity",
            "OnlineBackup",
            "DeviceProtection",
            "TechSupport",
            "StreamingTV",
            "StreamingMovies",
        ],
        "dependent_level": "No internet service",
    },
    "Sem telefone": {
        "trigger": ("PhoneService", "No"),
        "dependents": ["MultipleLines"],
        "dependent_level": "No phone service",
    },
}

print("Verificacao das regras estruturais nas 7043 linhas do dataset:\n")
for name, block in STRUCTURAL_BLOCKS.items():
    trigger_column, trigger_level = block["trigger"]
    triggered = raw[trigger_column] == trigger_level
    print(f"{name}: {trigger_column} == {trigger_level!r} -> {int(triggered.sum())} clientes")
    for dependent in block["dependents"]:
        holds = bool((raw.loc[triggered, dependent] == block["dependent_level"]).all())
        converse = bool(
            (
                raw.loc[raw[dependent] == block["dependent_level"], trigger_column] == trigger_level
            ).all()
        )
        print(
            f"    {dependent:18s} sempre {block['dependent_level']!r}: {holds}"
            f"   | reciproca: {converse}"
        )

print(
    f"\nTotal de regras estruturais verificadas: "
    f"{sum(len(b['dependents']) for b in STRUCTURAL_BLOCKS.values())}"
)

## 18. Explicabilidade local — explicando uma previsão individual

### Como o exemplo foi escolhido

O exemplo abaixo é **sintético e declarado antes de qualquer previsão**. Isso é
deliberado: escolher retrospectivamente um cliente do holdout cuja explicação ficasse
bonita seria selecionar um resultado depois de olhá-lo — exatamente o que este
trabalho evita em todas as outras seções.

Os dois perfis são os mesmos que a demonstração interativa do projeto oferece:

* **Perfil A** — cliente novo, fibra, contrato mensal, cheque eletrônico;
* **Perfil B** — cliente antigo, sem internet, contrato bienal, débito automático.

O perfil B também serve para ver o bloco estrutural da seção 17 em ação.

In [ ]:
SYNTHETIC_CUSTOMERS = {
    "A - cliente novo, fibra, contrato mensal": {
        "tenure": 2,
        "MonthlyCharges": 84.50,
        "TotalCharges": "169.00",
        "gender": "Female",
        "SeniorCitizen": 0,
        "Partner": "No",
        "Dependents": "No",
        "PhoneService": "Yes",
        "MultipleLines": "No",
        "InternetService": "Fiber optic",
        "OnlineSecurity": "No",
        "OnlineBackup": "No",
        "DeviceProtection": "No",
        "TechSupport": "No",
        "StreamingTV": "Yes",
        "StreamingMovies": "Yes",
        "Contract": "Month-to-month",
        "PaperlessBilling": "Yes",
        "PaymentMethod": "Electronic check",
    },
    "B - cliente antigo, sem internet, contrato bienal": {
        "tenure": 58,
        "MonthlyCharges": 24.90,
        "TotalCharges": "1444.20",
        "gender": "Male",
        "SeniorCitizen": 0,
        "Partner": "Yes",
        "Dependents": "Yes",
        "PhoneService": "Yes",
        "MultipleLines": "No",
        "InternetService": "No",
        "OnlineSecurity": "No internet service",
        "OnlineBackup": "No internet service",
        "DeviceProtection": "No internet service",
        "TechSupport": "No internet service",
        "StreamingTV": "No internet service",
        "StreamingMovies": "No internet service",
        "Contract": "Two year",
        "PaperlessBilling": "No",
        "PaymentMethod": "Bank transfer (automatic)",
    },
}


def explain(record: dict) -> dict:
    """Decompoe exatamente uma previsao nos 19 termos aditivos do logit."""
    frame = pd.DataFrame([record])[list(X_train.columns)]
    probability = float(final_model.predict_proba(frame)[0, 1])

    cleaned = final_model.named_steps["clean"].transform(frame)
    design = preprocessor.transform(cleaned)
    terms = (design * coefficients)[0]

    per_feature = {
        feature: float(sum(terms[i] for i, owner in enumerate(column_owner) if owner == feature))
        for feature in raw_features
    }
    model_logit = intercept + sum(per_feature.values())

    return {
        "probability": probability,
        "prediction": int(probability >= FROZEN_THRESHOLD),
        "model_logit": model_logit,
        "logit_error": abs(model_logit - float(final_model.decision_function(frame)[0])),
        "probability_error": abs(1.0 / (1.0 + np.exp(-model_logit)) - probability),
        "contributions": per_feature,
    }


for label, record in SYNTHETIC_CUSTOMERS.items():
    result = explain(record)
    decision = "SINALIZAR (risco de churn)" if result["prediction"] else "nao sinalizar"

    print("=" * 78)
    print(label)
    print("=" * 78)
    print(f"  probabilidade de churn : {result['probability']:.4f}")
    print(f"  limiar congelado       : {FROZEN_THRESHOLD:.4f}  (regra: probabilidade >= limiar)")
    print(f"  decisao                : {decision}")
    print("\n  Identidade da decomposicao:")
    print(f"    intercepto + soma das 19 contribuicoes = {result['model_logit']:+.10f}")
    print(f"    erro no logit        = {result['logit_error']:.3e}   (tolerancia 1e-12)")
    print(f"    erro na probabilidade= {result['probability_error']:.3e}")
    assert result["logit_error"] <= 1e-12 and result["probability_error"] <= 1e-12

    ranked = sorted(result["contributions"].items(), key=lambda kv: -abs(kv[1]))[:8]
    print("\n  Maiores contribuicoes (log-odds):")
    for feature, amount in ranked:
        arrow = "aumenta" if amount > 0 else ("reduz  " if amount < 0 else "neutro ")
        print(f"    {feature:20s} {str(record[feature])[:24]:26s} {amount:+8.4f}  {arrow}")
    print()

In [ ]:
# O bloco estrutural em acao no perfil B.
result_b = explain(SYNTHETIC_CUSTOMERS["B - cliente antigo, sem internet, contrato bienal"])
block = STRUCTURAL_BLOCKS["Sem internet"]
members = [block["trigger"][0]] + block["dependents"]
block_total = sum(result_b["contributions"][m] for m in members)

print("Perfil B - agregacao do bloco estrutural 'Sem internet'\n")
print("  Se apresentadas separadamente, pareceriam 7 razoes independentes:")
for member in members:
    print(f"    {member:20s} {result_b['contributions'][member]:+8.4f}")
print("\n  Agregadas em um unico fato ('este cliente nao tem internet'):")
print(f"    {'Sem internet (bloco)':20s} {block_total:+8.4f}")
print("\n  A soma e exata: o bloco continua particionando o logit.")

### O que uma contribuição é — e o que não é

Uma contribuição é o **termo aditivo daquela feature no preditor linear**: quantos
log-odds o valor daquela feature somou ao score, *nesta parametrização ajustada* e
*para este cliente*.

Ela **não** é:

* um **efeito causal** — nada aqui diz que mudar a feature mudaria o comportamento do
  cliente;
* um **contrafactual** — não é "se o contrato fosse anual, a probabilidade seria X";
* uma **recomendação** — "ofereça contrato anual" não segue de "contrato mensal
  contribuiu +0,7 log-odds".

A frase que o produto exibe, e que este relatório repete, é a formulação correta:

> As contribuições explicam o **cálculo do modelo** para esta entrada. Não são efeitos
> causais nem recomendações de mudança de comportamento do cliente.

## 19. Estratégia de monitoramento

Um modelo congelado não se degrada sozinho — **o mundo em volta dele muda**. O desenho
de monitoramento do projeto cobre oito frentes, organizadas do que é detectável
imediatamente para o que só é detectável com rótulos.

### 19.1 Qualidade de dados e esquema

Contadores por janela: requisições, registros, registros bem-sucedidos, esquema
inválido, valor de feature inválido, categoria não vista, violação estrutural.

O contrato de features é verificado a cada requisição. Ausência de coluna, tipo
inesperado ou valor em branco onde nenhuma regra permite são **rejeitados**, não
imputados silenciosamente. A regra de `TotalCharges` da seção 4 vive aqui: branco com
`tenure > 0` é erro, e permanece erro em produção.

### 19.2 Categorias não vistas

O encoder foi ajustado com `handle_unknown="ignore"`, então um plano novo ou meio de
pagamento novo **não derruba a predição** — vira um bloco de zeros e contribui nada.

Mas contribuir nada é uma decisão silenciosa: o modelo está pontuando um cliente
usando *menos informação do que aparenta*. Por isso a taxa de categorias não vistas é
monitorada explicitamente, com o rastreamento de valores distintos **limitado a 1024
por feature** para que um atacante não consiga fazer a memória crescer sem limite.

### 19.3 Consistência estrutural

As **sete regras** da seção 17 são verificadas a cada requisição. Um registro com
`InternetService = No` mas `OnlineSecurity = No` (em vez de `No internet service`) é
estruturalmente inconsistente: algum sistema a montante mudou.

As regras são **observadas, não impostas** — o registro ainda é pontuado. Impor
transformaria um sinal de monitoramento em uma falha de predição.

### 19.4 Drift das features

| Tipo | Métrica | Aviso | Crítico |
|---|---|---|---|
| Numérica | **PSI** (Population Stability Index) | 0,10 | 0,25 |
| Categórica | **TVD** (Total Variation Distance) | 0,10 | 0,25 |
| Fora da faixa de referência | proporção | 0,01 | 0,05 |

$$\text{PSI} = \sum_i (a_i - e_i)\ln\frac{a_i}{e_i}, \qquad \text{TVD} = \tfrac{1}{2}\sum_c |p_a(c) - p_e(c)|$$

O perfil de referência foi construído **do pool de treino**, nunca do holdout, e está
versionado com digest fixado.

### 19.5 Drift de predição

PSI sobre a distribuição de scores, mais a **taxa de positivos previstos** e sua
variação (aviso 0,05, crítico 0,10). Este é frequentemente o sinal que se move
primeiro: mudanças em várias features podem se compensar individualmente e ainda assim
deslocar o score.

### 19.6 O ponto metodológico que não pode ser omitido

> **PSI e TVD não são testes estatísticos.** São **distâncias descritivas** entre
> distribuições. Nenhum p-valor é calculado, nenhuma hipótese é testada.
>
> Os limiares 0,10 / 0,25 são uma **política operacional de monitoramento**
> (`OPERATIONAL_MONITORING_POLICY`), escolhida como heurística **antes de existir
> qualquer dado de produção**. Não são níveis de significância, não foram estimados
> dos dados, e **cruzar um deles não é evidência de degradação do modelo**. É um
> pedido de investigação.

Um cuidado adicional: janelas com menos de **100 registros** não emitem veredito. Uma
distância calculada sobre 12 observações é ruído com aparência de sinal.

### 19.7 Desempenho — só quando os rótulos chegam

Nenhuma métrica de desempenho (acurácia, recall, precisão, F1, ROC-AUC, AP, matriz de
confusão) é computada em produção nesta fase, **porque não existe verdade de campo em
produção**. Churn só é observável depois de uma janela de confirmação — tipicamente
semanas ou meses.

Calcular "acurácia em produção" sem rótulos exigiria inventá-los. O desenho declara
essa ausência em vez de mascará-la; quando os rótulos chegarem, as mesmas métricas do
holdout passam a ser calculáveis sobre coortes datadas.

### 19.8 Privacidade

O monitoramento agrega. Nenhum payload de requisição é persistido, nenhum
identificador de cliente é armazenado, e os contadores são por janela. O objetivo é
observar a **distribuição**, não os indivíduos.

## 20. Critérios de investigação e retreinamento

### O que este projeto deliberadamente não faz

**Não prescreve "retreinar todo mês".** Uma cadência fixa sem evidência é um ritual:
retreina quando nada mudou e não retreina quando algo muda na semana errada.

**Não implementa retreinamento automático.** Um sistema que se retreina sozinho pode
absorver corrupção de dados a montante como se fosse aprendizado, e substituir um
modelo validado por um não validado sem que ninguém decida.

### A escada de decisão

```
drift detectado (PSI/TVD acima do limiar)
        |
        v
INVESTIGAÇÃO  — o dado mudou, ou o mundo mudou?
                Bug a montante? Campanha? Mudança de mix?
        |
        +--> causa técnica  -> corrigir a origem. Não retreinar.
        |
        v
rótulos disponíveis para a coorte afetada
        |
        v
DIAGNÓSTICO   — o desempenho realmente caiu, medido com rótulos reais?
        |
        +--> não corroborado -> continuar observando. Não retreinar.
        |
        v
degradação persistente e relevante
        |
        v
CANDIDATO A RETREINAMENTO
        |
        v
PROTOCOLO COMPLETO DE VALIDAÇÃO — novo split, nova comparação, nova política
de calibração, novo limiar, novo holdout intocado, novo congelamento
        |
        v
promoção somente se o candidato passar
```

**Drift sozinho nunca justifica um retreinamento.** Ele justifica uma *investigação*.
A distinção é a diferença entre um sistema que responde a evidência e um que responde
a alarmes.

E o passo final não é negociável: um modelo candidato passa pelo **mesmo protocolo
inteiro** — incluindo um holdout novo e intocado. Um modelo promovido sem isso não tem
estimativa de desempenho válida, por melhor que pareça.

## 21. Demonstração da solução

O repositório inclui um produto funcional que embala este modelo: uma API de inferência
e uma página de demonstração interativa.

| Recurso | O que faz |
|---|---|
| `POST /api/v1/predict` | pontua um cliente; devolve probabilidade, decisão, limiar |
| `POST /api/v1/explain` | pontua **e** decompõe exatamente, com as contribuições da seção 18 |
| `GET /api/v1/portfolio` | métricas do holdout versionadas, com o caveat de exposição junto |
| `GET /api/v1/monitoring` | estado agregado de qualidade e drift da janela |
| `/demo` | formulário com as 19 features, score, decisão, explicação local, status |

A demonstração é **opt-in**: desligada por padrão, para que uma implantação que quer
apenas a API permaneça idêntica à que foi validada.

> **O que a interface é, e o que ela não é.** Ela é uma demonstração de que o modelo
> foi empacotado e é servível. Ela **não é evidência de que o modelo está correto**.
> Essa evidência vem inteiramente do protocolo experimental das seções 6 a 15 — da
> partição protegida, da comparação pareada, do limiar escolhido sem tocar no teste, e
> da única avaliação do holdout. Uma tela bonita com um modelo mal validado continua
> sendo um modelo mal validado.

## 22. Limitações

**1. Exposição do analista (seção 15).** A EDA precedeu a proteção formal do holdout.
A estimativa final pode carregar viés otimista não quantificável.

**2. Probabilidades não calibradas.** `calibration_policy = NONE`. Os scores são
posições de ordenação, não frequências. "0,42" não significa "42 % de chance".

**3. Nenhum custo real de negócio.** O limiar vem de uma política declarada
(maximização de F1), não de otimização de custo esperado. Sem custo de campanha, valor
de cliente e capacidade operacional, o limiar ótimo de negócio não pode ser calculado —
e não foi inventado.

**4. Nenhuma evidência causal.** Todo resultado é associativo. Contribuições explicam o
cálculo do modelo, não o comportamento do cliente.

**5. Nenhuma evidência de que o sistema reduz churn.** Isso exigiria um experimento de
intervenção — grupo de controle, campanha aleatorizada, medição de uplift. Nada disso
foi feito. O que se demonstra é **capacidade de predição**, não **efeito de negócio**.

**6. Um único dataset, um único recorte temporal.** O Telco Customer Churn é um retrato
estático, sem dimensão temporal explícita. Não há validação temporal, e nada garante
que os padrões se sustentem em outra operadora ou em outro período.

**7. Distribuição fixa.** Todo o trabalho assume que a população de produção se parece
com a de treino. É exatamente essa suposição que o monitoramento existe para vigiar.

## 23. Conclusão

### O modelo final

**Regressão logística** (`C = 1,0`, `solver = lbfgs`, sem `class_weight`) sobre as **19
features originais**, com `StandardScaler` nas 3 numéricas e `OneHotEncoder` nas 16
categóricas — 46 colunas transformadas, sem calibração, com limiar de decisão
**0,3272694566222328** e regra `probabilidade >= limiar`.

### Por que este modelo

Ele **venceu na métrica primária** (Average Precision) contra random forest e
histogram gradient boosting sob validação cruzada pareada; nenhum procedimento de
tuning cumpriu a regra de elegibilidade fixada antes dos resultados; e nenhum método de
calibração melhorou o Brier score de forma consistente.

Quando nenhum candidato justifica a substituição, o padrão pré-registrado é **manter o
modelo mais simples** — não adotar o de maior pontuação. O boosting ajustado teve AP
médio maior e mesmo assim não foi promovido, porque ganhava em 3 de 5 folds. Essa é a
decisão de engenharia mais importante deste trabalho.

Há um benefício adicional que a simplicidade compra: a decisão da logística é
**decomponível exatamente**. Cada previsão vem com a identidade
`intercepto + Σ contribuições = logit`, verificada a 1e-12 em toda explicação servida.

### Desempenho

No holdout de **1409 clientes**, aberto **uma única vez**:

| | |
|---|---|
| **Average Precision** | **0,634** (IC 95 % 0,578–0,686) — contra 0,265 sem informação |
| **ROC-AUC** | **0,842** (IC 95 % 0,818–0,863) |
| **Recall** | **0,722** — encontra ~72 % de quem sai |
| **Precisão** | **0,538** |
| **F1** | **0,616** |
| Acurácia | 0,762 (auxiliar; classe majoritária dá 0,735) |

### O efeito do limiar

Baixar de 0,50 para 0,327 leva o recall de ~0,54 para **~0,72** e a precisão de ~0,65
para **~0,54**, sinalizando ~35 % da carteira em vez de ~22 %. É uma troca deliberada:
mais clientes em risco encontrados, ao custo de mais contatos desnecessários e mais
capacidade operacional exigida.

### O que a explicabilidade mostrou

`tenure` domina a variação do score, seguido de `MonthlyCharges`, `InternetService`,
`Contract` e `TotalCharges`. `gender`, `Partner` e `PhoneService` são praticamente
inertes. As sete dependências estruturais foram identificadas e agregadas, de modo que
"este cliente não tem internet" apareça como **um fato**, não como sete evidências.

### Como seria operado

Monitoramento de qualidade de dados, categorias não vistas, consistência estrutural,
drift de features (PSI/TVD) e drift de predição, com desempenho medido **apenas quando
rótulos chegarem**. Drift dispara **investigação**, nunca retreinamento automático; um
modelo candidato repete o protocolo completo antes de qualquer promoção.

### A afirmação que este trabalho faz — e a que não faz

**Faz:** foi construído um sistema reprodutível de predição de churn, validado sob um
protocolo que protege o conjunto de teste, com limiar analisado, previsões explicadas
exatamente, inferência empacotada e monitoramento desenhado.

**Não faz:** não há evidência de que este sistema reduza churn. Isso exigiria um
experimento de intervenção que não foi conduzido. O que foi demonstrado é capacidade de
**identificar** clientes em risco — o que a operação faz com essa lista é outra
questão, e mediria outro resultado.

---

## Apêndice — reprodutibilidade e proveniência

### Âncoras verificadas nesta execução

| Item | Valor |
|---|---|
| SHA-256 do dataset bruto (canônico) | `88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a` |
| SHA-256 dos IDs de treino | `a553196dd46b672f6344867707144fbf56a8abe14b338a225662c7208a1450dd` |
| SHA-256 dos IDs do holdout | `1ad8aefb7e34776a78d76765d2465c630a41b3813b1d7d96d0d1b03d049776f6` |
| Limiar congelado | `0.3272694566222328` |
| Política de calibração | `NONE` |
| Regra de decisão | `probabilidade >= limiar` |

### Sementes e determinismo

Semente única `42` para a partição, para os folds de validação cruzada e para o
bootstrap. Sem seleção aleatória de hiperparâmetros, sem estimadores por amostragem na
explicabilidade.

### Fonte dos dados

Repositório público da IBM no GitHub, acessível por HTTPS sem autenticação. O conteúdo
é canonizado para terminadores `CRLF` e verificado por SHA-256 antes de qualquer
análise; divergência aborta a execução.

### Fronteira entre este notebook e o repositório

| | |
|---|---|
| **Repositório de engenharia** | fonte da verdade de produção: pipeline persistido, API, suíte de testes, monitoramento, registros determinísticos |
| **Este notebook** | reprodução acadêmica auto-contida do protocolo congelado, sem dependência do checkout local |

Ambos produzem os mesmos números — e cada número reproduzido aqui foi comparado, na
tela, com o valor registrado no artefato versionado correspondente.

In [ ]:
print("=" * 72)
print("EXECUCAO CONCLUIDA")
print("=" * 72)
print("  dataset verificado por SHA-256   : sim")
print("  particao identica a congelada    : sim")
print(f"  limiar reproduzido exatamente    : {selected_threshold == FROZEN_THRESHOLD}")
print(
    f"  metricas do holdout reproduzidas : "
    f"{all(abs(holdout_metrics[k] - FROZEN_HOLDOUT[k]) < 1e-6 for k in FROZEN_HOLDOUT)}"
)
print(f"  matriz de confusao reproduzida   : {observed_confusion == FROZEN_CONFUSION}")
print(f"  identidade da decomposicao       : erro maximo {max_error:.1e}")
print()
print("  academic_reproduction_only       : True")
print("  evaluated_configurations         : 1")
print("  selection_after_reproduction     : False")
print("  model_changed                    : False")
print()
for name, version in ENVIRONMENT.items():
    print(f"  {name:>14s} : {version}")